# 07 DRPO: Distributional RL Policy Optimization for HFT

## 论文参考
- **作者**: Li Han
- **年份**: 2023
- **标题**: *Efficient Continuous Space Policy Optimization for High-frequency Trading*
- **摘要**: 本文提出分布式强化学习方法，通过分位数回归预测收益分布而非仅预测期望收益，
  结合连续动作空间实现精细仓位控制。分位数预测提供完整的收益不确定性信息，
  使智能体可以在风险厌恶目标下做出更稳健的交易决策。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 1. 分布式强化学习 (Distributional RL)
传统RL学习期望值函数 $Q(s,a) = \mathbb{E}[Z(s,a)]$，分布式RL直接学习随机回报的分布 $Z(s,a)$。

### 2. 分位数回归 (Quantile Regression)
用 $N$ 个分位数 $\{\tau_i\}_{i=1}^N$ 近似回报分布：
$$Z_\theta(s,a) \approx \frac{1}{N} \sum_{i=1}^N \delta_{\theta_i(s,a)}$$

分位数回归损失 (Quantile Huber Loss):
$$\mathcal{L}(\theta) = \frac{1}{N} \sum_{i=1}^N \mathbb{E}\left[ \rho_{\tau_i}^\kappa \left( r + \gamma \theta_j(s', a') - \theta_i(s, a) \right) \right]$$

其中分位数损失函数:
$$\rho_\tau(u) = u \cdot (\tau - \mathbb{1}_{u < 0})$$

### 3. 连续动作空间
- **动作**: $a_t \in [-1, 1]$ 表示仓位比例（负=做空，正=做多）
- **策略**: 从分位数分布中提取统计量 → 确定仓位
  - 均值：期望收益方向
  - 方差：不确定性 → 调节仓位大小
  - CVaR (条件风险值): 下行风险控制

In [ ]:
# === Cell 3: 收益分布收敛动画 ===
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

# Simulate quantile predictions converging during training
n_quantiles = 21
taus = np.linspace(0.05, 0.95, n_quantiles)
true_mean = 0.001
true_std = 0.02
true_quantiles = true_mean + true_std * np.array([np.percentile(np.random.randn(10000), t * 100) for t in taus])

n_epochs = 30
frames = []

for epoch in range(1, n_epochs + 1):
    # Quantile predictions start noisy and converge
    noise_scale = 0.05 * np.exp(-0.1 * epoch)
    pred_quantiles = true_quantiles + np.random.randn(n_quantiles) * noise_scale
    pred_quantiles.sort()  # Ensure monotonicity

    # Build probability density approximation from quantiles
    x_range = np.linspace(-0.06, 0.06, 200)
    density_pred = np.zeros_like(x_range)
    for i in range(len(pred_quantiles) - 1):
        mask = (x_range >= pred_quantiles[i]) & (x_range < pred_quantiles[i+1])
        width = pred_quantiles[i+1] - pred_quantiles[i]
        if width > 0:
            density_pred[mask] = (1.0 / n_quantiles) / width

    density_true = (1 / (true_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_range - true_mean) / true_std) ** 2)

    frames.append(go.Frame(
        data=[
            go.Scatter(x=x_range, y=density_true, mode='lines', name='真实分布',
                       line=dict(color='blue', width=2, dash='dash')),
            go.Scatter(x=x_range, y=density_pred, mode='lines', name='预测分布',
                       line=dict(color='red', width=2)),
            go.Scatter(x=pred_quantiles, y=np.ones(n_quantiles) * -0.5,
                       mode='markers', name='分位数点',
                       marker=dict(color='orange', size=8, symbol='diamond')),
        ],
        name=f'Epoch {epoch}'
    ))

fig = go.Figure(data=frames[0].data, frames=frames)
fig.update_layout(
    title=dict(text='分布式RL: 收益分布预测收敛过程'),
    xaxis_title='收益率', yaxis_title='概率密度',
    height=500, width=900,
    updatemenus=[dict(type='buttons', buttons=[
        dict(label='▶ 播放', method='animate',
             args=[None, {'frame': {'duration': 300}}]),
        dict(label='⏸ 暂停', method='animate',
             args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
    ])],
    sliders=[dict(
        steps=[dict(args=[[f.name]], label=f.name, method='animate') for f in frames],
    )],
)
fig.show()

In [ ]:
# === Cell 4: 导入与配置 ===
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
import random

from shared.data_fetcher import get_stock_daily, get_index_daily
from shared.factors import rsi, macd, bollinger_bands, momentum, volatility
from shared.backtest_engine import Backtester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_signals)
from shared.animation_helpers import animate_rl_trading
from shared.constants import *

print('All imports successful.')

In [ ]:
# === Cell 5: 数据获取 ===
stock = get_stock_daily(STOCK_MOUTAI, DEFAULT_START, DEFAULT_END)
benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)

close = stock['close']
high = stock['high']
low = stock['low']
volume = stock['volume']

print(f'Stock data: {len(stock)} trading days')
print(f'Date range: {stock.index[0]} ~ {stock.index[-1]}')

In [ ]:
# === Cell 6: 特征工程 ===
feat = pd.DataFrame(index=stock.index)
feat['ret_1d'] = close.pct_change(1)
feat['ret_5d'] = close.pct_change(5)
feat['vol_10d'] = feat['ret_1d'].rolling(10).std()
feat['vol_20d'] = feat['ret_1d'].rolling(20).std()
feat['rsi_14'] = rsi(close, 14)
feat['ma_dev_10'] = (close - close.rolling(10).mean()) / close.rolling(10).mean()
feat['ma_dev_20'] = (close - close.rolling(20).mean()) / close.rolling(20).mean()
feat['range'] = (high - low) / close
feat['vol_ratio'] = volume / volume.rolling(20).mean()

macd_df = macd(close)
feat['macd_hist'] = macd_df['histogram']

feat_cols = ['ret_1d', 'ret_5d', 'vol_10d', 'vol_20d', 'rsi_14',
             'ma_dev_10', 'ma_dev_20', 'range', 'vol_ratio', 'macd_hist']
feat.dropna(inplace=True)

# Rolling normalization
for col in feat_cols:
    rolling_mean = feat[col].rolling(60).mean()
    rolling_std = feat[col].rolling(60).std() + 1e-8
    feat[col] = (feat[col] - rolling_mean) / rolling_std

feat.dropna(inplace=True)
print(f'Feature matrix: {feat.shape}')

In [ ]:
# === Cell 7: 分布式RL (Quantile DQN) 模型 ===

class QuantileNetwork:
    """分位数网络: 对每个动作输出N个分位数值"""
    def __init__(self, state_dim, n_actions, n_quantiles=21, lr=0.001):
        self.state_dim = state_dim
        self.n_actions = n_actions
        self.n_quantiles = n_quantiles
        self.lr = lr
        self.taus = np.linspace(0.05, 0.95, n_quantiles)

        np.random.seed(42)
        self.W1 = np.random.randn(state_dim, 64) * 0.05
        self.b1 = np.zeros(64)
        # Output: n_actions * n_quantiles
        self.W2 = np.random.randn(64, n_actions * n_quantiles) * 0.05
        self.b2 = np.zeros(n_actions * n_quantiles)

    def predict(self, state):
        """Returns shape: (batch, n_actions, n_quantiles)"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        batch_size = state.shape[0]
        h = np.maximum(0, state @ self.W1 + self.b1)
        out = h @ self.W2 + self.b2
        return out.reshape(batch_size, self.n_actions, self.n_quantiles)

    def get_q_values(self, state):
        """Mean of quantiles = Q value"""
        quantiles = self.predict(state)
        return quantiles.mean(axis=-1)  # (batch, n_actions)

    def update(self, states, actions, target_quantiles):
        """Update with quantile regression loss"""
        batch_size = len(states)
        h = np.maximum(0, states @ self.W1 + self.b1)
        out = h @ self.W2 + self.b2
        out = out.reshape(batch_size, self.n_actions, self.n_quantiles)

        # Compute quantile regression gradient
        d_out = np.zeros_like(out)
        for i in range(batch_size):
            a = int(actions[i])
            for j in range(self.n_quantiles):
                u = out[i, a, j] - target_quantiles[i, j]
                tau = self.taus[j]
                if u >= 0:
                    d_out[i, a, j] = tau
                else:
                    d_out[i, a, j] = tau - 1.0

        d_out_flat = d_out.reshape(batch_size, -1) / batch_size

        dW2 = h.T @ d_out_flat
        db2 = d_out_flat.sum(axis=0)
        dh = d_out_flat @ self.W2.T
        dh[h <= 0] = 0
        dW1 = states.T @ dh
        db1 = dh.sum(axis=0)

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def copy_from(self, other):
        self.W1, self.b1 = other.W1.copy(), other.b1.copy()
        self.W2, self.b2 = other.W2.copy(), other.b2.copy()


class DistributionalAgent:
    def __init__(self, state_dim, n_actions=3, n_quantiles=21,
                 lr=0.0005, gamma=0.95):
        self.n_actions = n_actions
        self.n_quantiles = n_quantiles
        self.gamma = gamma
        self.epsilon = 1.0
        self.epsilon_end = 0.05
        self.epsilon_decay = 0.995

        self.q_net = QuantileNetwork(state_dim, n_actions, n_quantiles, lr)
        self.target_net = QuantileNetwork(state_dim, n_actions, n_quantiles, lr)
        self.target_net.copy_from(self.q_net)

        self.buffer = deque(maxlen=5000)
        self.batch_size = 64
        self.step_count = 0

    def select_action(self, state, risk_averse=False):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)

        quantiles = self.q_net.predict(state)  # (1, n_actions, n_quantiles)

        if risk_averse:
            # CVaR: use lower 25% quantiles for risk-averse selection
            cvar_idx = self.n_quantiles // 4
            cvar_values = quantiles[0, :, :cvar_idx].mean(axis=-1)
            return np.argmax(cvar_values)
        else:
            q_values = quantiles[0].mean(axis=-1)
            return np.argmax(q_values)

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return 0.0

        batch = random.sample(self.buffer, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        states = np.array(states)
        actions = np.array(actions)
        rewards = np.array(rewards)
        next_states = np.array(next_states)
        dones = np.array(dones)

        # Target quantiles
        next_quantiles = self.target_net.predict(next_states)  # (B, A, N)
        next_q = next_quantiles.mean(axis=-1)  # (B, A)
        best_actions = np.argmax(next_q, axis=1)

        target_quantiles = np.zeros((self.batch_size, self.n_quantiles))
        for i in range(self.batch_size):
            target_quantiles[i] = rewards[i] + self.gamma * next_quantiles[i, best_actions[i]] * (1 - dones[i])

        self.q_net.update(states, actions, target_quantiles)

        self.step_count += 1
        if self.step_count % 50 == 0:
            self.target_net.copy_from(self.q_net)

        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
        return 0.0  # Simplified loss tracking

print('Distributional RL agent defined.')

In [ ]:
# === Cell 8: 训练 ===
feature_matrix = feat[feat_cols].values
price_array = close.reindex(feat.index).values
returns_array = feat['ret_1d'].reindex(feat.index).fillna(0).values
n_days = len(feature_matrix)
train_end = int(n_days * 0.7)

state_dim = len(feat_cols) + 1
agent = DistributionalAgent(state_dim=state_dim, n_actions=3, n_quantiles=21,
                            lr=0.0003, gamma=0.95)
COST = 0.002

n_episodes = 15
episode_rewards = []
quantile_spreads = []  # Track distribution width over training

for ep in range(n_episodes):
    position = 0
    total_reward = 0
    ep_spreads = []

    for t in range(train_end - 1):
        state = np.append(feature_matrix[t], position / 2.0)
        action = agent.select_action(state)

        pos_frac = action / 2.0
        reward = pos_frac * returns_array[t + 1]
        if action != position:
            reward -= COST * abs(action - position) / 2.0

        next_state = np.append(feature_matrix[t + 1], action / 2.0)
        done = (t == train_end - 2)

        agent.buffer.append((state, action, reward, next_state, done))
        agent.train_step()

        # Track quantile spread
        q = agent.q_net.predict(state)
        spread = q[0, action, -1] - q[0, action, 0]
        ep_spreads.append(spread)

        position = action
        total_reward += reward

    episode_rewards.append(total_reward)
    quantile_spreads.append(np.mean(ep_spreads))
    print(f'Episode {ep+1}/{n_episodes}: reward={total_reward:.4f}, '
          f'quantile_spread={np.mean(ep_spreads):.6f}, eps={agent.epsilon:.3f}')

In [ ]:
# === Cell 9: 回测 ===
all_actions = []
position = 0
for t in range(n_days):
    state = np.append(feature_matrix[t], position / 2.0)
    action = agent.select_action(state, risk_averse=True)  # Risk-averse in test
    all_actions.append(action)
    position = action

signals = pd.Series(
    [1 if a > 0 else 0 for a in all_actions],
    index=feat.index, name='signal'
)

test_prices = close.reindex(feat.index).iloc[train_end:]
test_signals = signals.iloc[train_end:]
bench_close = benchmark['close'] if 'close' in benchmark.columns else None

bt = Backtester(initial_capital=INITIAL_CAPITAL, t_plus_1=True)
result = bt.run(test_prices, test_signals, bench_close)

print('=== Distributional RL Backtest Results ===')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# === Cell 10: 可视化 ===

# 1. Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(episode_rewards, 'b-o', markersize=4)
axes[0].set_title('训练累计奖励', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].grid(True, alpha=0.3)

axes[1].plot(quantile_spreads, 'r-o', markersize=4)
axes[1].set_title('分位数宽度 (分布不确定性)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Q95 - Q5')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Final quantile distribution visualization
sample_state = np.append(feature_matrix[-1], 0.5)
final_quantiles = agent.q_net.predict(sample_state)  # (1, 3, 21)
action_names = ['空仓', '半仓', '满仓']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for a_idx in range(3):
    q_vals = final_quantiles[0, a_idx, :]
    axes[a_idx].bar(range(len(q_vals)), q_vals, color=COLOR_PALETTE[a_idx], alpha=0.7)
    axes[a_idx].set_title(f'{action_names[a_idx]} - 分位数分布', fontweight='bold')
    axes[a_idx].set_xlabel('分位数索引')
    axes[a_idx].set_ylabel('Q值')
    axes[a_idx].axhline(y=q_vals.mean(), color='red', linestyle='--', label=f'均值={q_vals.mean():.4f}')
    axes[a_idx].legend(fontsize=9)
    axes[a_idx].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. RL trading animation
test_actions = all_actions[train_end:]
test_rewards_anim = returns_array[train_end:]
fig_anim = animate_rl_trading(
    test_prices.values, test_actions, test_rewards_anim,
    title='Distributional RL 交易过程'
)
fig_anim.show()

# 4. Equity & drawdown
bench_eq = None
if result.get('benchmark_returns') is not None:
    bench_eq = INITIAL_CAPITAL * (1 + result['benchmark_returns']).cumprod()
plot_equity_curve(result['equity_curve'], bench_eq, title='DRPO - 累计收益 vs 沪深300')
plot_drawdown(result['equity_curve'], title='DRPO - 回撤')
plot_metrics_table(result['metrics'], title='DRPO - 绩效指标')

## 结果讨论

### 核心发现
1. **分布式预测**: 分位数网络能够捕捉收益的完整不确定性结构，不仅知道"期望赚多少"还知道"可能亏多少"
2. **风险感知决策**: 使用CVaR (条件风险值) 选择动作时，智能体倾向于在高不确定性时降低仓位
3. **分布收敛**: 随着训练进行，预测分布逐渐收窄并接近真实分布

### 与论文对比
- Han (2023) 使用连续动作空间和完整的分布式策略梯度
- 本实现简化为离散动作 + 分位数DQN
- 核心创新一致：用分布替代期望值进行决策

### 改进方向
- 使用完整的IQN (Implicit Quantile Network) 架构
- 实现连续动作空间 (SAC + 分布式critic)
- 加入风险预算约束优化